<a href="https://colab.research.google.com/github/DAS-Bodhi/Fed-Recc/blob/main/Securing_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Privacy Preserving Ad Recc System using Fed Learning

In [ ]:
!pip install opacus -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 10.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch.func import functional_call, vmap, grad
from opacus.accountants import RDPAccountant
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
GPU available: True


#***Historical/Technical Context on libraries***

#***Pytorch***

1.	It was inspired by Torch library(written in lua) which had tensor operations and some autograd support.
2.	Introduced Dynamic computation graphs as opposed to Static ones.
3.	This means, you had to 1st defining a graph and then run it.
4.	You could not specify any conditions without using special graph-specific conditional operations (`tf.cond`, `tf.while_loop`) with their own syntax, essentially a separate mini-language for control flow. But now, `if`, `for`, `while` operate on real tensor values in real time, the graph naturally adapts to code. This is precisely why 'for epoch in range(300):' training loop works.
5.	Trying to print(z) in Phase 1 would show you something like `Tensor("add:0", shape=(), dtype=float32)` — a description of the node, not an actual number, to actually see intermediate values, you had to explicitly add them to the `sess.run(...)` fetch list and re-run the whole session.
6.	Graph structure used to be defined 1st, if inputs could vary the structure, you had to use padding.
7.	Since graph construction and execution were separate steps, an error triggered during `sess.run(...)` might point to a problem in how the graph was built, but the stack trace often didn't cleanly map back to the specific line of your Python code that caused it.
8.	The "layer as an object" idea was popularized by frameworks like Torch (Lua) and Caffe (2013), where a network was literally built by stacking pre-defined layer objects.

# ***Pandas***
1.	"Panel Data", an econometrics term for exactly this kind of structured, labeled tabular data.
2.	Structured tables — pandas' core object, the DataFrame, is a table with named columns and (usually) numbered rows, conceptually identical to a spreadsheet or a SQL table.
3.	Functionality on top of that structure — this is the real value pandas adds.
4.	"train_df[train_df['user_idx'].isin(users_in_client)]" = "give me only the rows matching this condition."
5.	Column-wise transformation: "df['user_idx'] = df['user_id'].map(user2idx)" — apply a transformation to an entire column at once (vectorized), rather than looping row by row in plain Python
6.	Reading/writing structured files
7.	plain NumPy array is just numbers in a grid. pandas sits on top of NumPy (literally built using it internally) and adds the "spreadsheet-like" layer.




#***Random***
1.	Its core algorithm is the Mersenne Twister, a pseudo-random number generator.
2.	notable for an extremely long period before repeating (2^19937 − 1, an almost incomprehensibly large number)
3.	not cryptographically secure, only for general purpose.


#***Pytorch and torch.nn module***

#***Layer 1: tensors + autograd + the computational graph***

***Tensors:***
Tensors are the data structure — GPU-capable, multi-dimensional arrays, functionally similar to NumPy arrays but with one crucial addition: they can track "requires_grad=True"

***Tensor Tracking:***

1.	It records every operation performed on that tensor into the computational graph
2.	It reserves a slot, called ".grad", to eventually store that tensor's gradient. ".backward()" computes it and fills up the slot.
3.	If "requires_grad=False", PyTorch doesn't bother recording the graph at all for operations involving that tensor — no memory spent, because there was never any intention of computing a gradient for it.
4.	"with torch.no_grad():" (used in "evaluate()" function) is a performance win, since you're never going to call ".backward()" inside that block so no memory allocation for it.
5.	The five parameters — "user_emb.weight", "item_emb.weight", "user_bias.weight", "item_bias.weight", "global_bias" — automatically has "requires_grad=True" set on it because, "nn.Embedding" and "nn.Parameter" both set this flag internally, by design, since the entire point of these classes is "this thing is meant to be learned."

***Model Tracking and parameter registration (enabled by super().__init__()):***

operates at the level of the whole model as a collection. It's nn.Module's own separate bookkeeping system — a registry that keeps track of which tensors belong to this model, so that model.parameters(), model.state_dict(), model.to(device) (it enables all 3) can find and act on all of them together, as a group.

***The computational graph :***

1.	It is what gets built when operations that have requires_grad=True are performed on tensors. Every operation — a multiplication, a lookup, a sum, an addition — gets recorded as a node BTS, remembering what produced it and what simple local derivative rule applies to that specific operation.
2.	.backward() walks this recorded graph in reverse, applying the chain rule at each node and composing them together — computing gradients for every connected parameter in one pass.

#***Layer 2: organizing all of this into reusable pieces (torch.nn)***

***Containers:***
a holding structure for multiple layers/modules, built specifically so that PyTorch's registration system can see and track everything inside it.
`self.layers = [nn.Linear(10, 10), nn.Linear(10, 10), nn.Linear(10, 10)]` - a list. A generic Python object. The registry has no special awareness that inside this list are three more nn.Module objects that should also be tracked
vs
`self.layers = nn.ModuleList([nn.Linear(10, 10), nn.Linear(10, 10), nn.Linear(10, 10)])` - it knows to announce every module it holds, so all three nn.Linear layers do get properly registered and discovered by .parameters().

1.	***nn.Sequential —*** holds layers and automatically chains them in forward().
2.	***nn.ModuleList —*** holds layers, registered correctly, but does not automatically chain them — still write forward() is needed to decide exactly how/when each one gets called (useful when the order or logic isn't a simple straight pipeline — e.g., calling the same layer multiple times in a loop, or branching).
3.	***nn.ModuleDict —*** same as ModuleList, but keyed by name instead of position — useful when you want to look up a specific layer by a meaningful string name rather than an index number.


***Why nn.linear is not used in the baseline?***

Matrix Factorization's whole design is that users and items never directly interact except through a dot product — a very constrained, specific form of interaction. NCF (the model I dropped) replaces that dot product with one or more nn.Linear layers — letting the model learn a much more flexible, arbitrary function of the concatenated user+item vectors, rather than being locked into "similarity via dot product."

***Convolutional Layers (nn.Conv1d/nn.Conv2d):***

1.	Usecase- Signal Processing (Image recognition/identification etc.)
2.	Convolution is a specific mathematical operation between two functions (or two grids/sequences of numbers) that combines them by sliding one across the other and computing a weighted sum at each position.
3.	A convolutional layer slides a small grid of learnable numbers (the filter or kernel — say, 3×3) across the input image, one position at a time. At each position, it multiplies the filter's values with whatever pixels it's currently sitting over, sums the result into one number, and moves on. Do this across the whole image, and you get a new grid of outputs called a feature map.
4.	stacking many of these layers, with many filters each, lets the network build from simple patterns to increasingly complex ones.

***RNN/LSTM (nn.RNN, nn.LSTM, nn.GRU) — solving sequence***

1.	some data has order that matters — a sentence, a time series, a sequence of clicks. "nn.Linear" has no built-in notion of this.
2.	RNNs' idea: process one element at a time, carrying forward a "hidden state" summarizing everything seen so far.
3.	plain RNNs suffer from the vanishing gradient problem — during backpropagation through time (the same chain rule, applied across many sequential steps), gradients from early steps get multiplied by many small numbers in a row and shrink toward zero, meaning the model effectively "forgets" long-range dependencies. LSTM remarkably, a full 20+ years before Transformers — introduced gating mechanisms specifically designed to let gradients flow backward through many time steps without vanishing.

***The nn class vs. nn.functional***

nn.X (class) — creates a persistent object that can hold onto data between calls (like nn.Linear's weight matrix, which must persist somewhere). Set up once, called many times.
nn.functional.x (function) — a plain, stateless calculation. Nothing is remembered between calls; every input has to be passed in explicitly, every time.
***Why both exist:***
some operations (like a layer's weights) genuinely need to persist as an object across many calls. Others (like a one-off loss calculation) don't need any persistent state at all, so a plain function is simpler.

Where this actually mattered in code: DP-SGD's per-sample gradients needed your model's parameters treated as plain, swappable function inputs (via torch.func's vmap) — which is exactly why that had to be built the "functional" way, not because gradients wouldn't otherwise track, but because vectorizing per-sample computation needs parameters passed explicitly, not locked inside a stateful object.

***nn.init — the deeper history, tying back to vanishing/exploding gradients***

Xavier/Glorot initialization (Glorot & Bengio, 2010) was the first widely-adopted "smart" initialization scheme — it scales the randomness based on how many inputs and outputs a layer has, specifically to keep activations from shrinking toward zero or exploding toward infinity as they pass through many stacked layers. Before this, deep networks were notoriously hard to train, partly because naive fixed-scale initialization (like your std=0.01, applied identically regardless of layer size) works fine for a shallow model like yours, but causes real problems in deep networks — a small std that's fine for one layer compounds badly across 20 stacked layers.

Kaiming/He initialization (He et al., 2015) refined this further, specifically accounting for ReLU activation's behavior (which zeroes out negative values, changing the math slightly from Xavier's assumptions). This was published alongside the famous "ResNet" paper, which needed very deep networks (100+ layers) to train reliably.

Why none of this applies to you, precisely: your model has exactly one layer of real computation (the dot product) — no depth for activations to shrink or explode across. A fixed small std=0.01 is sufficient and simpler; reaching for Xavier/Kaiming would be solving a problem (instability across many stacked layers) that a shallow MF model structurally doesn't have.

#***CUDA - Compute Unified Device Architecture***

•	CUDA is NVIDIA's own brand name for their GPU programming platform, lets programmers write code that runs on NVIDIA GPUs, first released in 2006/2007.
•	AMD has its own separate, competing platform (called ROCm), and PyTorch has separate support for that, accessed completely differently.
•	Google Colab specifically provisions NVIDIA GPUs (like the T4 you enabled earlier) precisely because CUDA has become the dominant, most widely-supported platform in deep learning — most frameworks (PyTorch, TensorFlow) were built with CUDA as the primary target.
•	torch.device("cuda") is choosing CUDA specifically as the destination label, creates a small Python object. It's not a number, not a memory address, not a connection — it's a lightweight object whose entire content is essentially just the string "cuda". This object tells other functions where to send or place something.
Why we need this as a separate object, rather than just writing "cuda" directly everywhere
We can write the string directly in many places (tensor.to("cuda") works too) — torch.device(...) mainly exists so you can build the choice once, store it in a variable, and reuse that same decision everywhere consistently.


In [ ]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

cols = ["user_id", "item_id", "rating", "timestamp"]
df = pd.read_csv("ml-100k/u.data", sep="\t", names=cols)
print(df.shape)
df.head()

--2026-09-12 08:54:18--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
  Issued certificate has expired.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  9.27MB/s    in 0.5s    

2026-09-12 08:54:19 (9.27 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test  

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


#***Dataset - MOVIELENS***
We're reading `u.data` — which has no column structure of its own — and loading its contents into a new, separate DataFrame object, df, telling pandas as we do so what to call each column and how the file's values are separated. The naming/structuring happens entirely in memory, on this new df object — never on the original file itself.

A DataFrame is pandas' main data structure — a table, with labeled rows and labeled columns, capable of holding multiple different data types across different columns. `df` isn't a plain Python list or dict — it's a distinct, purpose-built object type, with capabilities neither of those has. Columns can have different Data types (int, float etc.).

***Pros***

•	Established benchmark — maintained by GroupLens Research (University of Minnesota), the standard recommender-systems benchmark for over two decades, directly comparable to a large body of prior FL-recommender work using this same dataset.

•	Clean and ethically unambiguous — freely available for research, no consent controversy, explicit (not inferred) ratings on a clear 1-5 scale.

•	Right-sized for the actual research question — the dissertation's focus is privacy attacks, not state-of-the-art accuracy; a small, fast-to-train dataset allows rapid iteration (DP-SGD, FedProx, secure aggregation tested across days, not weeks).

***Cons***

•	No natural client structure — the data was collected centrally, not from genuinely distributed devices.

•	Scale mismatch — real FL deployments (e.g. Gboard) involve millions of small clients; this simulation (10 clients, thousands of ratings each) sits closer to cross-silo FL than cross-device FL.

•	Atypically high density — real federated recommenders tend to have sparser, noisier per-client signal than MovieLens provides.

•	No temporal/connectivity realism — real clients go online/offline asynchronously; this simulation runs all clients synchronously every round.

•	Coarse demographic metadata (u.user: age, gender, occupation, zip) — usable for a toy property-inference demo, but not richly reflective of more sensitive real-world attributes.

No publicly available dataset provides a naturally federated (rather than artificially partitioned) collaborative filtering task; this is a recognized limitation across FL research generally (Kairouz et al., 2021), not specific to this dissertation's dataset choice

#***Concrete Metrics***
Users	= 943

Items	= 1,682

Ratings	= 100,000

Sparsity	= 93.6953% (density ≈ 6.3%)

Ratings per user	= min 20, max 737, mean 106, std 101

Ratings per item	= min 1, max 583, mean 59, std 80

Rating scale	= 1-5 (explicit)

Rating distribution	= 1: 6.1% · 2: 11.4% · 3: 27.2% · 4: 34.2% · 5: 21.2%

Collection period	Sept 1997 - April 1998

•	93.7% sparsity is notably lower than real production recommenders (often 99.9%+) — worth naming as a limitation on generalizability.

•	Mean 106 / std 101 (per-user ratings) signals a wide, right-skewed distribution — relevant if testing whether MIA success correlates with how much data a user contributed.

•	1997-98 collection date — pre-streaming era rating norms, worth a one-line caveat.

#***Alternatives for future use***

***MovieLens 1M/10M/20M***

Same structure, larger scale (original proposal's choice)	Same synthetic-partition limitation, just slower — reasonable as future work

***Netflix Prize***

Larger scale, thematically fitting privacy history	No longer officially distributed, due to its own re-identification litigation
Criteo	Realistic industrial scale and sparsity	Different domain (ad clicks), heavy feature engineering — real scope risk on this timeline

***Amazon Reviews / Yelp***

Larger scale, temporal/geographic structure	Less standardized as an FL-recsys benchmark

***LEAF benchmark suite (Caldas et al., 2018)***

Purpose-built for realistic FL benchmarking	No recommender-specific dataset included (targets vision/text)

In [ ]:
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()

n_users = len(user_ids)
n_items = len(item_ids)
print(f"n_users={n_users}, n_items={n_items}")

user2idx = {}
for i, u in enumerate(user_ids):
    user2idx[u] = i

item2idx = {}
for i, it in enumerate(item_ids):
    item2idx[it] = i

df['user_idx'] = df['user_id'].map(user2idx)
df['item_idx'] = df['item_id'].map(item2idx)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

df[['user_id', 'user_idx', 'item_id', 'item_idx']].head(10)

n_users=943, n_items=1682


,user_id,user_idx,item_id,item_idx
0,196,0,242,0
1,186,1,302,1
2,22,2,377,2
3,244,3,51,3
4,166,4,346,4
5,298,5,474,5
6,115,6,265,6
7,253,7,465,7
8,305,8,451,8
9,6,9,86,9


***Pre-processing (Re-indexing and Splitting)***

For users- We first extract the unique user IDs into an array, then print its length. Next, we build a dictionary, user2idx, where each raw user ID is mapped to an index based on the order it first appears in that array — not sorted, just first-come-first-numbered. Then we create a new column, user_idx, by using that dictionary to look up each existing value in the user_id column and writing the corresponding index back into the DataFrame.

We did the exact thing for the items as well.
Finally, we split the (now re-indexed) data into train and test sets.


In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        dot = (self.user_emb(user) * self.item_emb(item)).sum(dim=1)
        return (dot + self.user_bias(user).squeeze()
                     + self.item_bias(item).squeeze()
                     + self.global_bias)

`self.user_emb` is an attribute of the model -- not an object OF `__init__`,
but something created and assigned by code running inside `__init__` (the method Python automatically runs once, when the model is constructed). It holds an `nn.Embedding` object: the full 943x64 lookup table, one row
per user, each row a 64-number vector.

Every one of the 60,352 individual cells in that table is independently drawn from a normal distribution with mean 0 (the default, since mean was not specified) and standard deviation 0.01:

$$w_{i,j} \sim \mathcal{N}(\mu=0,\ \sigma=0.01)$$

Note this is standard deviation, not variance (variance would be $\sigma^2 = 0.01^2 = 0.0001$, a much tighter spread.)

**Biases: no symmetry problem at all**

Biases start at exact zero, and that's fine there was never a symmetry problem for biases, in either sense.

No INTERNAL symmetry issue: a bias is a single scalar per user, with no internal dimensions to differentiate from each other.

No EXTERNAL (cross-user) symmetry issue either: even if two users started with identical bias values, their gradients naturally diverge from the first training step, since each user's gradient depends on their own prediction error:

$$\frac{\partial \mathcal{L}}{\partial b_u} = 2\,e_{u,i}, \qquad e_{u,i} = \hat{r}_{u,i} - r_{u,i}$$

Different users have different ratings, hence different $e_{u,i}$, hence naturally different gradients, regardless of shared starting
values.

**Embeddings: internal symmetry problem**

Cross-user or external symmetry was never a problem here either, for the same data-driven reason as biases, different users' embeddings would diverge over training even if initialized identically.

The real issue is INTERNAL: within a single user's own 64-dimensional vector. For one fixed rating, the error term is identical across every
dimension $k$: $$\frac{\partial \mathcal{L}}{\partial p_{u,k}} = 2,e_{u,i}, q_{i,k}$$

$p_u$ — user $u$'s embedding vector. This is exactly `self.user_emb(user)` in your code — the 64-number vector representing that specific user's learned taste.

$q_i$ — item $i$'s embedding vector. This is `self.item_emb(item)` — the 64-number vector representing that specific item's learned characteristics.

$p_{u,k}$ and $q_{i,k}$ — the individual $k$-th number inside those vectors.

The only thing that can make dimension $k=1$'s gradient differ from dimension $k=2$'s gradient is the ITEM's embedding value at that same
dimension, $q_{i,k}$. If both $p_u$ and $q_i$ were initialized uniformly (e.g. all-zero), every dimension would receive mathematically
IDENTICAL gradients forever:

$$\frac{\partial \mathcal{L}}{\partial p_{u,1}} = \frac{\partial \mathcal{L}}{\partial p_{u,2}} = \cdots = \frac{\partial \mathcal{L}}{\partial p_{u,64}} \quad \text{if } q_{i,1} = q_{i,2} = \cdots = q_{i,64}$$

**Why random init fixes this, and only this**

Random initialization exists specifically to fix this internal, cross-dimension problem. The fact that it ALSO gives different users different starting vectors is a harmless side effect of randomness being applied cell-by-cell across the whole tensor, not a deliberate fix for a cross-user problem, since that problem never existed in the first place.

**Per-user/per-item updates during training**

Once training begins, gradients flow back ONLY to the specific rows actually looked up in a given batch so each user's embedding and bias update independently, driven entirely by that user's own accumulated prediction errors:

$$\theta_{\text{row } u} \leftarrow \theta_{\text{row } u} - \eta \sum_{(u,i) \in \text{batch}} \frac{\partial \mathcal{L}_{u,i}}{\partial \theta_{\text{row } u}}$$

Parameters for users who appear rarely in the data change correspondingly rarely.

1. `user_emb` — [943, 64], each user's taste vector
2. `item_emb` — [1682, 64], each item's vector
3. `user_bias` — [943, 1], each user's rating tendency
4. `item_bias` — [1682, 1], each item's rating tendency
5. `global_bias` — [1], one shared average (nn.Parameter, not nn.Embedding, since it's a single value, not a lookup table)

**Forward** - the prediction function, called automatically. Takes a *batch* of user/item indices (not single values) and computes all predictions in that batch at once:

$$\hat{r}_{u,i} = b_0 + b_u + b_i + \mathbf{p}_u \cdot \mathbf{q}_i$$
	​
`.squeeze()` — strips an unwanted trailing dimension. `self.user_bias(user)` returns shape [batch_size, 1]; dot is shape [batch_size]. Without squeezing, adding them would broadcast into an unintended [batch_size, batch_size] grid instead of adding element-by-element.

In [ ]:
class TunableRecommender(nn.Module):
    """
    hidden_layers=0 reproduces the exact MatrixFactorization model above
    (dot product). hidden_layers>=1 concatenates user+item embeddings and
    passes them through that many shared dense layers before predicting --
    moving progressively toward an NCF-style architecture. This is the
    direct test of the sparse-gradient hypothesis: MF's per-example
    gradient touches only 2 embedding rows; shared dense layers make
    EVERY example's gradient touch ALL of those shared weights.
    """
    def __init__(self, n_users, n_items, emb_dim=64, hidden_layers=0, hidden_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

        self.hidden_layers = hidden_layers
        if hidden_layers == 0:
            self.mlp = None
        else:
            layers = []
            input_dim = emb_dim * 2
            for _ in range(hidden_layers):
                layers += [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
                input_dim = hidden_dim
            layers.append(nn.Linear(input_dim, 1))
            self.mlp = nn.Sequential(*layers)

    def forward(self, user, item):
        p_u = self.user_emb(user)
        q_i = self.item_emb(item)

        if self.hidden_layers == 0:
            interaction = (p_u * q_i).sum(dim=-1)
        else:
            concat = torch.cat([p_u, q_i], dim=-1)
            interaction = self.mlp(concat).squeeze(-1)

        return (interaction + self.user_bias(user).squeeze()
                            + self.item_bias(item).squeeze()
                            + self.global_bias)

In [ ]:
# Helper function: measures model performance on any given dataset split
def evaluate(model, df_split):
    model.eval()  # switch to eval mode (no functional effect here, but standard practice)

    # convert the given DataFrame's columns into tensors on the GPU/CPU
    u = torch.tensor(df_split['user_idx'].values, dtype=torch.long, device=device)
    i = torch.tensor(df_split['item_idx'].values, dtype=torch.long, device=device)
    r = torch.tensor(df_split['rating'].values, dtype=torch.float32, device=device)

    with torch.no_grad():  # don't track gradients — we're only measuring, not training
        preds = model(u, i)  # calls forward() from Cell 4 — one prediction per row
        # compute MSE, take its square root to get RMSE, then unwrap the 1-element
        # tensor into a plain Python float using .item()
        rmse = torch.sqrt(nn.functional.mse_loss(preds, r)).item()

    return rmse  # hand back the plain float RMSE value


# Create the actual model instance (first real object built from the Cell 4 blueprint)
model = MatrixFactorization(n_users, n_items, emb_dim=64).to(device)  # .to(device) moves all params to GPU/CPU

# Set up the training tools
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
# model.parameters() hands the optimizer every learnable piece (works because of
# super().__init__() back in Cell 4, which registered them)

loss_fn = nn.MSELoss()  # reusable MSE loss object, called every epoch below


# Convert the training data into tensors ONCE, outside the loop (reused every epoch)
u_train = torch.tensor(train_df['user_idx'].values, dtype=torch.long, device=device)
i_train = torch.tensor(train_df['item_idx'].values, dtype=torch.long, device=device)
r_train = torch.tensor(train_df['rating'].values, dtype=torch.float32, device=device)


# Early stopping tracking variables, initialized before training starts
best_rmse = float('inf')   # starts impossibly high, so epoch 0 always counts as "best so far"
best_state = None          # will hold a snapshot of the model's best weights
patience = 15               # how many non-improving epochs to tolerate before giving up
bad_epochs = 0               # running counter of consecutive non-improving epochs


# Main training loop
for epoch in range(300):  # epoch takes values 0, 1, 2, ... up to 299 (or until early stop)

    model.train()             # switch to training mode
    optimizer.zero_grad()     # clear gradients left over from the previous epoch

    preds = model(u_train, i_train)   # forward pass — all 80,000 ratings predicted at once
    loss = loss_fn(preds, r_train)     # compute this epoch's MSE loss

    loss.backward()    # autograd computes gradients for all 168,626 parameters
    optimizer.step()    # Adam uses those gradients to actually update every parameter

    # check performance on the held-out test set, AFTER this epoch's update
    test_rmse = evaluate(model, test_df)   # calls the function defined at the top

    if test_rmse < best_rmse:
        # new best found — save the RMSE, snapshot the current weights, reset the bad-streak counter
        best_rmse, best_state, bad_epochs = test_rmse, model.state_dict(), 0
    else:
        # no improvement this epoch
        bad_epochs += 1
        if bad_epochs >= patience:
            # 15 non-improving epochs in a row — stop training early
            print(f"Early stopping at epoch {epoch}, best RMSE={best_rmse:.4f}")
            break   # exits the for loop immediately, skipping remaining epochs

    # progress logging every 10th epoch (cosmetic only, doesn't affect training)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: test_RMSE={test_rmse:.4f}")


# after the loop ends: restore the BEST weights ever seen, not necessarily the last ones
model.load_state_dict(best_state)

print(f"\nCentralized benchmark RMSE: {best_rmse:.4f}")

Epoch 0: test_RMSE=3.6705
Epoch 10: test_RMSE=3.0084
Epoch 20: test_RMSE=1.3683
Epoch 30: test_RMSE=1.2806
Epoch 40: test_RMSE=0.9833
Epoch 50: test_RMSE=0.9934
Epoch 60: test_RMSE=0.9455
Epoch 70: test_RMSE=0.9353
Epoch 80: test_RMSE=0.9387
Early stopping at epoch 85, best RMSE=0.9353

Centralized benchmark RMSE: 0.9353


#Cell 5 — Centralized Training Loop (Non-Federated Baseline)

This cell trains the Matrix Factorisation model on the full training
set at once (no federation yet) -- establishing a reference RMSE that
the later FedAvg result gets compared against.

--- Loss function (MSE), computed once per epoch on the full batch ---

$$\mathcal{L} = \frac{1}{N}\sum_{n=1}^{N} (\hat{r}_n - r_n)^2, \qquad N = 80000$$

Called as: loss = loss_fn(preds, r_train)

#Evaluation metric (RMSE), computed only inside evaluate()

$$\text{RMSE} = \sqrt{\mathcal{L}} = \sqrt{\frac{1}{N}\sum_{n=1}^{N} (\hat{r}_n - r_n)^2}$$

RMSE puts the error back into the original rating scale (an RMSE of
0.98 means predictions are off by about 0.98 stars, on average).
RMSE is never used to compute gradients -- only MSE is differentiated;
RMSE exists purely for human-readable reporting.

#Gradient, computed by loss.backward()

For every parameter theta (every number inside user_emb, item_emb,
user_bias, item_bias, global_bias):

$$g_t = \frac{\partial \mathcal{L}}{\partial \theta}$$

This is a DIRECTION, not a step -- it tells you which way increases
loss, so training moves the opposite way.

#Adam's parameter update, applied by optimizer.step()

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\, g_t \qquad \text{(smoothed direction)}$$

$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\, g_t^2 \qquad \text{(smoothed magnitude squared)}$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \qquad \text{(bias correction)}$$

$$\theta_{t+1} = \theta_t - \eta\, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

lr=0.01 is eta; weight_decay=1e-5 adds a small penalty discouraging
any parameter from growing too large (mild overfitting defense).

#Early stopping condition, checked every epoch

$$\text{if } \; \text{RMSE}_{\text{test}}^{(t)} \geq \min_{k \le t} \big(\text{RMSE}_{\text{test}}^{(k)}\big) \; \text{for 15 consecutive epochs} \rightarrow \text{stop}$$

Uses strict less-than for improvement -- a flat plateau still counts
as "no improvement" and increments the patience counter.

#Final restored weights

$$\theta^{*} = \theta_{t^*}, \qquad t^* = \arg\min_t \; \text{RMSE}_{\text{test}}^{(t)}$$

model.load_state_dict(best_state) restores the BEST epoch's weights,
not necessarily the last epoch -- since training can drift slightly
worse before early stopping catches up (15-epoch patience window).

#Result

This produces the centralized benchmark RMSE (~ 0.98), the reference
point Cell 9's FedAvg result (~ 1.08) gets compared against to
quantify the accuracy cost of federation.

In [ ]:
def dirichlet_partition(train_df, n_clients, alpha, n_items, seed=42):
    """
    Non-IID partition via Dirichlet distribution (Li et al. and similar
    FL literature): for each item, draw a Dirichlet distribution over
    which clients get how much of that item's ratings. Small alpha ->
    highly skewed/non-IID. Large alpha -> closer to IID.
    """
    rng = np.random.default_rng(seed)
    client_dfs = [[] for _ in range(n_clients)]

    for item in range(n_items):
        item_rows = train_df[train_df['item_idx'] == item]
        if len(item_rows) == 0:
            continue
        proportions = rng.dirichlet(alpha=[alpha] * n_clients)
        shuffled = item_rows.sample(frac=1, random_state=seed).reset_index(drop=True)
        split_points = [0] + list((np.cumsum(proportions) * len(shuffled)).astype(int))
        split_points[-1] = len(shuffled)
        for c in range(n_clients):
            start, end = split_points[c], split_points[c + 1]
            client_dfs[c].append(shuffled.iloc[start:end])

    return [pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame() for dfs in client_dfs]


N_CLIENTS = 10
client_data = dirichlet_partition(train_df, N_CLIENTS, alpha=0.5, n_items=n_items)

for client_id, df_c in enumerate(client_data):
    print(f"Client {client_id}: {len(df_c)} ratings")

Client 0: 7725 ratings
Client 1: 8181 ratings
Client 2: 8335 ratings
Client 3: 8195 ratings
Client 4: 7443 ratings
Client 5: 7810 ratings
Client 6: 7694 ratings
Client 7: 7979 ratings
Client 8: 7883 ratings
Client 9: 8755 ratings


# Cell 6 — Non-IID Partition via Dirichlet Distribution

This cell splits 943 users' ratings across N_CLIENTS=10 simulated clients, using a Dirichlet distribution to make the split genuinely non-IID (uneven), rather than a random, statistically-similar split.

# Dirichlet distribution

A normal distribution (nn.init.normal_, used for embeddings) generates a single random number. A Dirichlet distribution generates a random set of numbers that always sum to exactly 1 — a random probability vector.

$$\text{Dir}(\alpha_1, \alpha_2, \ldots, \alpha_k) \;\rightarrow\; (p_1, p_2, \ldots, p_k), \qquad \sum_{c=1}^{k} p_c = 1$$

In this code: rng.dirichlet(alpha=[alpha] * n_clients) draws exactly one such vector — one proportion per client (10 numbers that sum to 1).

# alpha parameter controls

alpha is passed as the same value repeated for every client ([alpha] * n_clients) — this is called a "symmetric" Dirichlet. It controls how uneven the drawn proportions tend to be:

$$\alpha \to 0 \quad\Rightarrow\quad \text{extremely skewed (e.g. } p = [0.97, 0.01, 0.00, \ldots, 0.02]\text{)}$$

$$\alpha \to \infty \quad\Rightarrow\quad \text{nearly equal (e.g. } p \approx [0.1, 0.1, \ldots, 0.1]\text{)}$$

Small alpha: one or two clients dominate a given item's ratings almost entirely — highly non-IID, mirroring how a niche movie might only be popular among a narrow slice of real users. Large alpha: proportions flatten out toward equal shares — approaching the original IID random-shuffle partition from the simplified baseline version. alpha=0.5 (used here) sits at a moderate skew, a common choice in FL heterogeneity literature (Li et al. and similar).

# Why per-ITEM, not per-user

The whole partition is built by looping over every item (1682 total), not every user. For each item, a fresh Dirichlet draw decides how that specific item's ratings split across the 10 clients — so different items can be skewed toward completely different clients. This simulates realistic heterogeneity: which movies a client's users happen to like varies client to client, exactly as it would on real distributed devices.

# The full per-item mechanism, traced through

Take one item with 6 ratings, 3 clients (simplified example):

1. Pull only this item's rows:
item_rows = train_df[train_df['item_idx'] == item]   # 6 rows

2. Shuffle them into random order (breaks any hidden ordering bias):
shuffled = item_rows.sample(frac=1).reset_index(drop=True)

Position:  0    1    2    3    4    5

User:     512   88   340   7   901   45

3. Draw this item's proportions from the Dirichlet distribution:
proportions = [0.5, 0.3, 0.2]   # sums to 1

4. Convert proportions into actual cut points along the 6 shuffled rows:
$$\text{cumsum}([0.5, 0.3, 0.2]) = [0.5,\ 0.8,\ 1.0]$$

$$\times 6 \text{ rows} = [3.0,\ 4.8,\ 6.0] \;\rightarrow\; \text{int} \;\rightarrow\; [3, 4, 6]$$

$$\text{split points} = [0, 3, 4, 6]$$

5. Slice the shuffled rows at those cut points, one slice per client:
Client 0: positions 0-3 (3 rows) -> shuffled.iloc[0:3]
Client 1: positions 3-4 (1 row)  -> shuffled.iloc[3:4]
Client 2: positions 4-6 (2 rows) -> shuffled.iloc[4:6]

| Position | 0   | 1  | 2   | 3 | 4   | 5  |
|----------|-----|----|-----|---|-----|----|
| User ID  | 512 | 88 | 340 | 7 | 901 | 45 |
| Client   | 0   | 0  | 0   | 1 | 2   | 2  |

6. Each client's slice is appended to that client's own running list:
client_dfs[0].append(...), client_dfs[1].append(...), etc.

# Repeated across all 1682 items

Steps 1-6 repeat once per item. Each repetition draws a fresh, independent Dirichlet proportion vector — so item 0 might favor clients [0, 2], while item 1 favors clients [4, 7], entirely unrelated. By the end, client_dfs[c] holds 1682 small chunks (one per item, possibly empty for some), one growing list per client.

# Final step: stitching the chunks together

return [pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        for dfs in client_dfs]

For each client, its 1682 small chunks get concatenated into one single, complete DataFrame — like sewing many small patches into one finished piece of cloth, separately for each of the 10 clients. The if dfs else pd.DataFrame() guards against a client that, by chance, received zero rows from every single item (more likely at very small alpha).

# Why shuffle before slicing (step 2) is necessary

Without shuffling, "first 3 rows go to client 0" would always mean "the first 3 users in whatever order they appear in the raw data" — if that raw order correlates with anything (e.g. user ID, signup time), client assignment would silently inherit that bias. Shuffling first ensures which user ends up where is random, even though client sizes (via alpha) are deliberately uneven.

# End result

10 DataFrames, client_data[0] through client_data[9], each holding a genuinely non-IID slice of the 80,000 training ratings — some clients richer in certain items than others, mirroring real federated heterogeneity rather than a random, statistically-uniform split.

In [ ]:
def local_train_dp_prox(global_state_dict, client_df, n_users, n_items,
                          emb_dim=64, hidden_layers=0, hidden_dim=64, local_epochs=5, lr=0.01,
                          clip_norm=1.0, noise_multiplier=1.0, batch_size=256,
                          mu=0.1):
    # global_state_dict: current global model's weights, sent to this client
    # client_df: THIS client's own data only (from Cell 6's Dirichlet partition)
    # clip_norm: max allowed L2 norm for any single example's gradient (DP-SGD)
    # noise_multiplier: controls how much Gaussian noise gets added (DP-SGD)
    # mu: FedProx's proximal strength -- how hard to penalize drifting from global model

    # build a fresh, isolated local model, then load the CURRENT global weights into it
    local_model = TunableRecommender(n_users, n_items, emb_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
    local_model.load_state_dict(global_state_dict)

    # convert this model's parameters into a plain dict, since torch.func needs
    # parameters passed explicitly as function inputs, not locked inside a stateful object
    params = {k: v.detach().clone() for k, v in local_model.named_parameters()}
    buffers = {k: v.detach().clone() for k, v in local_model.named_buffers()}

    # frozen snapshot of the GLOBAL params used only as FedProx's comparison
    # point, never updated during local training (that's what makes it "global")
    global_params_frozen = {k: v.detach().clone() for k, v in params.items()}
    param_names = list(params.keys())  # the 5 parameter names: user_emb.weight, etc.

    # convert THIS client's own ratings into tensors (not the full train_df)
    users_t = torch.tensor(client_df['user_idx'].values, dtype=torch.long, device=device)
    items_t = torch.tensor(client_df['item_idx'].values, dtype=torch.long, device=device)
    ratings_t = torch.tensor(client_df['rating'].values, dtype=torch.float32, device=device)
    n_samples = len(client_df)  # how many ratings this client has

    # defines the loss for ONE single example (no batch dimension)
    # this function gets vmapped below, so it only ever needs to handle 1 sample at a time
    def compute_loss(params, buffers, user, item, rating):
        u, i, r = user.unsqueeze(0), item.unsqueeze(0), rating.unsqueeze(0)  # add back a batch dim of size 1
        pred = functional_call(local_model, (params, buffers), (u, i))  # forward pass using explicit params
        task_loss = ((pred - r) ** 2).mean()  # ordinary MSE, for this one example

        # FedProx term: penalize how far current params have drifted from
        # the frozen global params same for every sample, since it only
        # depends on params, not on (user, item, rating)
        prox_term = sum(((params[name] - global_params_frozen[name]) ** 2).sum()
                         for name in param_names)
        return task_loss + (mu / 2.0) * prox_term  # combined DP-SGD + FedProx loss

    # builds a function that computes compute_loss's gradient for EVERY
    # sample in a batch simultaneously (vmap = vectorized map)
    per_sample_grad_fn = vmap(grad(compute_loss), in_dims=(None, None, 0, 0, 0))
    # in_dims: params/buffers are shared (None = not batched), user/item/rating ARE batched (0)

    for epoch in range(local_epochs):  # only 5 local epochs, not 300 -- short on-device training
        perm = torch.randperm(n_samples)  # shuffle this client's data fresh, every epoch

        for start in range(0, n_samples, batch_size):  # walk through in mini-batches
            idx = perm[start:start + batch_size]  # this mini-batch's shuffled indices
            b_users, b_items, b_ratings = users_t[idx], items_t[idx], ratings_t[idx]  # fetch this batch's data
            actual_bs = len(idx)  # actual batch size (last batch may be smaller)
            if actual_bs == 0:
                continue  # skip if somehow an empty batch occurs

            # compute one gradient PER EXAMPLE in this batch, all at once
            per_sample_grads = per_sample_grad_fn(params, buffers, b_users, b_items, b_ratings)

            # compute each example's gradient NORM, combined across all 5 parameter tensors
            sq_norms = torch.zeros(actual_bs, device=device)  # one running total per example
            for name in param_names:
                g = per_sample_grads[name]  # shape [actual_bs, ...] -- this param's per-example gradients
                flat = g.reshape(actual_bs, -1)  # flatten everything except the batch dimension
                sq_norms += (flat ** 2).sum(dim=1)  # add this param's contribution to each example's squared norm
            per_sample_norms = torch.sqrt(sq_norms + 1e-12)  # sqrt to get actual norm (tiny epsilon avoids /0)

            # clipping factor per example: never let any single example's
            # gradient exceed clip_norm, whatever its raw size was
            clip_factors = torch.clamp(clip_norm / per_sample_norms, max=1.0)  # cap at 1.0 (never SCALE UP)

            # apply clipping, sum across the batch, add noise, average, and update
            for name in param_names:
                g = per_sample_grads[name]  # this param's per-example gradients again
                shape = g.shape[1:]  # this param's shape, excluding the batch dimension
                factor = clip_factors.view(-1, *([1] * len(shape)))  # reshape clip factors to broadcast correctly
                clipped_sum = (g * factor).sum(dim=0)  # scale each example's grad, then sum across the batch

                # calibrated Gaussian noise -- the actual DP mechanism, added to the SUMMED gradient
                noise = torch.normal(
                    mean=0.0, std=noise_multiplier * clip_norm, size=clipped_sum.shape, device=device
                )
                noisy_grad = (clipped_sum + noise) / actual_bs  # average over the batch, like a normal gradient step
                params[name] = params[name] - lr * noisy_grad  # manual gradient descent update (no optimizer object)

    #copy the final trained params back into a real model object, so we can return a normal state_dict() the same way local_train did before ---
    with torch.no_grad():
        for name, p in local_model.named_parameters():
            p.copy_(params[name])

    return local_model.state_dict(), n_samples  # return updated weights + this client's data count (for FedAvg weighting)

# Cell 7 — Local Client Training (DP-SGD + FedProx combined)

This function simulates one client's on-device training for one FedAvg round. It combines three mechanisms at once: FedProx (stability under non-IID data), per-sample gradient clipping, and calibrated Gaussian noise (together forming DP-SGD).

# Where the input parameters come from (this was a key confusion point)

`global_state_dict` is NOT related to Cell 5's centralized model — Cell 5 and Cell 7/9 are completely separate experiments. `global_state_dict` comes from Cell 9's own `global_model`, which starts fully random (Cell 4's `nn.init` logic, run once at round 0), then gets progressively updated round by round through this exact cycle: Cell 7 (local training) -> Cell 8 (secure aggregation) -> loaded back into Cell 9's `global_model` -> sent out again next round.

# What mu (FedProx strength) actually does

```python
prox_term = sum(((params[name] - global_params_frozen[name]) ** 2).sum() for name in param_names)
return task_loss + (mu / 2.0) * prox_term
```

`prox_term` measures how far the current local parameters have drifted from the frozen global parameters (squared difference, summed across all 168,626 numbers). `mu` controls how strongly that drift gets penalized in the final loss:

$$\mu = 0 \Rightarrow \text{no penalty at all — ordinary DP-SGD, client can drift freely}$$
$$\mu \text{ large} \Rightarrow \text{client is forced to stay very close to the global model, barely learning from its own data}$$

This matters specifically because Cell 6's Dirichlet partition makes client data genuinely non-IID — without `mu` holding clients back toward the global model, 10 clients training on very different data could drift in wildly different directions, producing a confused, poor result once averaged in Cell 8.

# Is task_loss "cut up" MSE — and is it per-item?

Yes — `task_loss` is computed for exactly ONE rating at a time (per-item), not a full-batch average. This is because `compute_loss` is written to handle a single example, then `vmap` (explained below) runs it across many examples in parallel — it is never manually batch-averaged inside the function itself.

One subtlety worth knowing: `prox_term` is IDENTICAL for every sample in a batch (it only depends on current vs. global params, not on any specific rating) — yet it still gets included in the per-sample gradient that DP-SGD clips. This is a deliberate implementation simplification: strictly speaking, Abadi et al.'s original DP-SGD clips only data-dependent gradients, while a global regularizer like the proximal term is sometimes handled separately. Worth naming as an honest design-choice note in the dissertation write-up.

# What vmap actually is

`vmap` = "vectorized map." It takes a function written for ONE example and automatically runs it across many examples simultaneously (in parallel), instead of requiring a manual Python for-loop.

```python
per_sample_grad_fn = vmap(grad(compute_loss), in_dims=(None, None, 0, 0, 0))
```

`in_dims=(None, None, 0, 0, 0)` tells vmap how to treat each argument:
- `params`, `buffers` -> `None` = shared, NOT batched (same for every example)
- `user`, `item`, `rating` -> `0` = batched along dimension 0 (a different value per example)

Without vmap, you'd need a slow Python loop, computing one gradient at a time, sequentially — vmap does this in parallel instead, which is both faster and is the actual mechanism that lets per-sample gradients coexist with a model whose forward() takes two inputs (user, item) — the exact thing that made Opacus's automatic wrapping incompatible earlier.

Important: vmap only COMPUTES the 256 per-sample gradients — it does not update any parameters. The actual parameter update happens only once per batch, after clipping + summing + noise + averaging all 256 gradients together.

# Where does 256 (or any batch number) come from

There is no hardcoded 256 — it comes from the `batch_size` parameter (default 256 in the function signature). `n_samples` (this client's total ratings, e.g. 8000) gets split into chunks of `batch_size`, so training happens in small groups rather than all at once (memory constraint, since per-sample gradients need one full gradient tensor per example, per parameter, held in memory simultaneously).

# What torch.randperm(n_samples) does

`n_samples` is a plain integer (e.g. 8000), but `torch.randperm(n)` returns a tensor containing `0` to `n-1`, shuffled into random order — every index appears exactly once, just reordered.

```python
torch.randperm(8)  # e.g. tensor([3, 0, 6, 1, 7, 4, 2, 5])
```

This does NOT shuffle the actual data — `users_t`, `items_t`, `ratings_t` stay in their original order the whole time. Instead, `perm` is used as a set of shuffled INDICES: `users_t[idx]` reads the data in shuffled order without physically moving anything, the same way reading books off a shelf in a randomly chosen order doesn't require rearranging the shelf.

# What range(0, n_samples, batch_size) does

`range(start, stop, step)` generates a sequence of starting positions for each batch:

```python
range(0, 8000, 256)  # -> 0, 256, 512, 768, ..., up to (but not including) 8000
```

Each value becomes the starting point of one batch: `idx = perm[start:start+batch_size]` slices out that batch's shuffled indices. The final batch may be smaller than `batch_size` if `n_samples` doesn't divide evenly — Python's slicing handles this automatically without erroring.

# What gradient norm means

A gradient isn't a single number — it's one value per parameter (168,626 of them). The "norm" collapses all of them into a single number representing overall magnitude:

$$\|g\| = \sqrt{\sum_k g_k^2}$$

```python
sq_norms += (flat ** 2).sum(dim=1)   # square and sum every parameter's contribution
per_sample_norms = torch.sqrt(sq_norms + 1e-12)   # square root -> the actual norm; tiny epsilon avoids divide-by-zero
```

This is needed because DP-SGD clips based on how LARGE an individual example's gradient is — you need one number per example to decide how much to shrink it, if at all.

# How the Gaussian noise is calibrated (not arbitrary)

```python
noise = torch.normal(mean=0.0, std=noise_multiplier * clip_norm, size=clipped_sum.shape, device=device)
```

The standard deviation is deliberately set to `noise_multiplier * clip_norm`, not a fixed arbitrary number — this is the exact formula from Abadi et al. (2016):

$$\sigma = \text{noise\_multiplier} \times \text{clip\_norm}$$
$$\text{noise} \sim \mathcal{N}(0, \sigma^2)$$

Why multiply by clip_norm specifically: clipping guarantees no single example's gradient exceeds `clip_norm` — this is the "sensitivity" DP theory requires noise to be calibrated against. If noise were a fixed number unrelated to clip_norm, it would either be way too small (no real privacy, if gradients are naturally large) or way too large (destroys all signal, if gradients are naturally small). Scaling by clip_norm keeps the noise proportional to the actual bound on any one example's influence.

`noise_multiplier` is the separate dial controlling HOW MUCH privacy you want (larger = more noise = stronger privacy, smaller epsilon) — this is exactly the same `noise_multiplier` value later fed into `compute_epsilon()` for the formal privacy accounting.

# Clipping mechanism, tied together

```python
clip_factors = torch.clamp(clip_norm / per_sample_norms, max=1.0)
```

For an example with norm 5.0 (and clip_norm=1.0): factor = min(1, 1.0/5.0) = 0.2 -> gradient gets shrunk to exactly clip_norm.
For an example with norm 0.3 (already small): factor = min(1, 1.0/0.3) = 1.0 -> unchanged, since `max=1.0` ensures gradients are only ever shrunk, never amplified.

# summary

One client receives the current global weights, trains briefly (5 local epochs, not 300 — deliberately short, since FedAvg's whole point is limited local computation between communication rounds, and long local training on a small, non-IID client dataset risks heavy overfitting and drifting the global model apart). Each individual rating's gradient is computed separately (via vmap), clipped to bound its influence, summed, given calibrated noise, and averaged into a single per-batch parameter update — repeated across all batches and epochs. FedProx's proximal term, baked into the loss itself, discourages the client from drifting too far from the global model throughout this whole process. Only the final updated weights (never raw ratings) are returned, along with how many ratings this client had (used later for FedAvg's weighted averaging).

In [ ]:
def generate_pairwise_masks(n_clients, param_shapes, seed=None, device=None):
    """
    For every pair of clients (i, j), generate one shared random mask per
    parameter. Client i ADDS the mask, client j SUBTRACTS the same mask --
    so when all clients' masked updates are summed, every pairwise mask
    cancels out exactly, and only the true sum survives. The server never
    sees any individual client's real update, only the total.
    """
    if seed is not None:
        torch.manual_seed(seed)
    masks = {name: [torch.zeros(shape, device=device) for _ in range(n_clients)] for name, shape in param_shapes.items()}
    for i in range(n_clients):
        for j in range(i + 1, n_clients):
            for name, shape in param_shapes.items():
                m = torch.randn(shape, device=device)
                masks[name][i] = masks[name][i] + m
                masks[name][j] = masks[name][j] - m
    return masks


def client_mask_update(client_state, weight, masks, client_id):
    """
    CLIENT-SIDE: pre-multiply by this client's FedAvg weight BEFORE
    masking. Masks only cancel under a PLAIN sum, so weighting must
    happen before masking, not after -- doing it after breaks the
    cancellation.
    """
    masked = {}
    for key, tensor in client_state.items():
        masked[key] = tensor.float() * weight + masks[key][client_id]
    return masked


def secure_aggregate(masked_client_updates):
    """
    SERVER-SIDE: plain sum only. The server never applies weights itself --
    they were already applied client-side, before masking.
    """
    avg_state = {}
    for key in masked_client_updates[0].keys():
        avg_state[key] = sum(update[key] for update in masked_client_updates)
    return avg_state

# Cell 8 — Secure Aggregation (Pairwise Additive Masking)

Server-side counterpart to Cell 7. Combines all 10 clients' updates into one new global model, without the server ever seeing any individual client's real update — only the sum.

# Why pairwise masking at all

A single mask on one client would corrupt the final sum (the mask never cancels). The fix: every PAIR of clients shares one random mask — one client adds it, the other subtracts it — so when both are summed, the mask vanishes:

$$(\text{real}_0 + m) + (\text{real}_1 - m) = \text{real}_0 + \text{real}_1$$

With 10 clients, every possible pair (45 total pairs) gets its own independent mask, so every client's individual update stays hidden, while the final total still comes out exactly correct.

# Full mathematical proof that the final result is correct

Notation: $n=10$ clients. For every pair $(i,j)$ with $i<j$, a random mask $m_{ij}$ is drawn. Convention: the smaller index ADDS the mask, the larger index SUBTRACTS it.

Step 1 — client $c$'s total mask contribution:

$$M_c = \sum_{j>c} m_{cj} \;-\; \sum_{i<c} m_{ic}$$

Step 2 — sum all clients' masks together. Every $m_{ij}$ (for $i<j$) appears in this double sum exactly twice: once positive (when $c=i$) and once negative (when $c=j$):

$$\sum_{c=0}^{9} M_c = \sum_{i<j} m_{ij} - \sum_{i<j} m_{ij} = 0$$

All masks cancel exactly, regardless of what random values they took.

Step 3 — client-side, weighting happens BEFORE masking:

$$\text{masked}_c = w_c \cdot \theta_c + M_c, \qquad w_c = \frac{N_c}{N}$$

Step 4 — server-side, plain sum only (no weights applied server-side):

$$\text{avg\_state} = \sum_{c=0}^{9} \text{masked}_c = \sum_{c=0}^{9} \left( w_c \theta_c + M_c \right)$$

Step 5 — split the sum:

$$\text{avg\_state} = \underbrace{\sum_{c=0}^{9} w_c \theta_c}_{\text{real FedAvg result}} + \underbrace{\sum_{c=0}^{9} M_c}_{=\,0 \text{ (Step 2)}} = \sum_{c=0}^{9} \frac{N_c}{N}\theta_c$$

This is exactly the FedAvg formula (McMahan et al.) -- the masks add zero net effect, and the true weighted average survives.

# Why the ORDER (weight before mask) is mathematically required

If masking happened before weighting instead:

$$\text{masked}_c = w_c \cdot (\theta_c + M_c) = w_c \theta_c + w_c M_c$$

Summing across clients:

$$\sum_c w_c \theta_c + \sum_c w_c M_c$$

Now $\sum_c w_c M_c \neq 0$ in general -- because each mask $m_{ij}$'s two copies (positive and negative) get multiplied by DIFFERENT weights ($w_i$ and $w_j$):

$$w_i \cdot m_{ij} - w_j \cdot m_{ij} = (w_i - w_j) \cdot m_{ij} \neq 0 \quad \text{whenever } w_i \neq w_j$$

This is exactly the bug found and fixed during testing -- the first version applied weighting after masking, producing wrong results (e.g. 1.4677 instead of the correct 2.0), because client data sizes (and therefore weights) differed.

#summary

Every pairwise mask cancels EXACTLY when summed plainly across all clients (Step 2) -- but this cancellation only survives if weighting is applied before masking, since applying different weights to a mask's two paired copies afterward breaks the cancellation (Step 6). The client never reveals its real update; the server never sees anything but the correct final weighted sum.

In [ ]:
def get_client_data_at_round(client_full_df, round_idx, num_rounds, global_min_ts, global_max_ts):
  progress = (round_idx + 1) / num_rounds
  cutoff_ts = global_min_ts + progress * (global_max_ts - global_min_ts)
  return client_full_df[client_full_df['timestamp'] <= cutoff_ts]
  """
  Simulates temporal realism using the dataset's REAL timestamps. Instead
  of every client having its full slice of data available from round 0,
  each client only sees ratings whose timestamp has actually been
  "reached" by this round.
  """
  """
  def compute_epsilon(n_samples, batch_size, noise_multiplier, epochs, delta=1e-5):
    steps = epochs * (n_samples // batch_size)
    sample_rate = batch_size / n_samples
    accountant = RDPAccountant()
    for _ in range(steps):
        accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
    eps, _ = accountant.get_privacy_spent(delta=delta)
    return eps"""
def compute_epsilon(n_samples, batch_size, noise_multiplier, epochs, delta=1e-5):
    steps = epochs * (n_samples // batch_size)
    sample_rate = batch_size / n_samples
    accountant = RDPAccountant()
    for _ in range(steps):
        accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
    alphas = [1 + x / 10.0 for x in range(1, 100)] + list(range(12, 64))
    eps, _ = accountant.get_privacy_spent(delta=delta, alphas=alphas)
    return eps


In [ ]:
import time

results = {}

for HIDDEN_LAYERS in [6, 7, 8, 9, 10]:
    print(f"\n{'='*20} hidden_layers={HIDDEN_LAYERS} {'='*20}")

    # Setup, before the loop
    client_data_full = dirichlet_partition(train_df, N_CLIENTS, alpha=0.5, n_items=n_items)
    global_min_ts = train_df['timestamp'].min()
    global_max_ts = train_df['timestamp'].max()

    global_model = TunableRecommender(n_users, n_items, emb_dim=64, hidden_layers=HIDDEN_LAYERS, hidden_dim=64).to(device)
    param_shapes = {name: p.shape for name, p in global_model.named_parameters()}

    NUM_ROUNDS = 300
    round_times = []

    # Main FedAvg loop
    for rnd in range(NUM_ROUNDS):
        round_start = time.time()
        global_state = global_model.state_dict()
        client_states, client_sizes = [], []

        for client_full_df in client_data_full:
            # temporal realism: this client only sees ratings whose real
            # timestamp has been "reached" by this round
            available_df = get_client_data_at_round(client_full_df, rnd, NUM_ROUNDS, global_min_ts, global_max_ts)
            if len(available_df) == 0:
                continue  # this client has no data yet at this point in real time

            # DP-SGD + FedProx local training, on this client's currently-available data
            state, n_samples = local_train_dp_prox(
                global_state, available_df, n_users, n_items,
                hidden_layers=HIDDEN_LAYERS, hidden_dim=64,
                clip_norm=1.0, noise_multiplier=4.5, mu=0.1
            )
            client_states.append(state)
            client_sizes.append(n_samples)

        if len(client_states) == 0:
            continue  # no client has data yet this early -- skip aggregation

        # secure aggregation: weight client-side, mask, then plain-sum server-side
        total_size = sum(client_sizes)
        masks = generate_pairwise_masks(len(client_states), param_shapes, device=device)
        masked_updates = [
            client_mask_update(client_states[i], client_sizes[i] / total_size, masks, i)
            for i in range(len(client_states))
        ]
        new_global_state = secure_aggregate(masked_updates)
        global_model.load_state_dict(new_global_state)
        round_times.append(time.time() - round_start)

        if rnd % 10 == 0 or rnd == NUM_ROUNDS - 1:
            rmse = evaluate(global_model, test_df)
            print(f"Round {rnd}: test_RMSE={rmse:.4f}, last round took {round_times[-1]:.2f}s")

    # Final privacy accounting, for THIS hidden_layers configuration
    final_rmse = evaluate(global_model, test_df)
    eps = compute_epsilon(n_samples=int(np.mean(client_sizes)), batch_size=256,
                           noise_multiplier=4.5, epochs=5 * NUM_ROUNDS)

    results[HIDDEN_LAYERS] = {"rmse": final_rmse, "epsilon": eps, "round_times": round_times}

    print(f"\nhidden_layers={HIDDEN_LAYERS} FINAL: RMSE={final_rmse:.4f}, epsilon={eps:.2f}")

# ---- Summary across all three configurations, once everything finishes ----
print("\n" + "="*50)
print("SUMMARY -- sparse-gradient hypothesis test")
print("="*50)
for hl, r in results.items():
    print(f"hidden_layers={hl}: RMSE={r['rmse']:.4f}, epsilon={r['epsilon']:.2f}")


==================== hidden_layers=6 ====================
Round 0: test_RMSE=3.5905, last round took 1.14s
Round 10: test_RMSE=2.7805, last round took 1.67s
Round 20: test_RMSE=1.5777, last round took 2.69s
Round 30: test_RMSE=1.1387, last round took 4.24s
Round 40: test_RMSE=1.1674, last round took 4.23s
Round 50: test_RMSE=1.1756, last round took 4.41s
Round 60: test_RMSE=1.1798, last round took 4.85s
Round 70: test_RMSE=1.1727, last round took 6.21s
Round 80: test_RMSE=1.1747, last round took 8.49s
Round 90: test_RMSE=1.1755, last round took 10.13s
Round 100: test_RMSE=1.1761, last round took 11.43s
Round 110: test_RMSE=1.1750, last round took 12.10s
Round 120: test_RMSE=1.1728, last round took 12.97s
Round 130: test_RMSE=1.1655, last round took 13.84s
Round 140: test_RMSE=1.1724, last round took 14.68s
Round 150: test_RMSE=1.1704, last round took 15.12s
Round 160: test_RMSE=1.1698, last round took 16.33s
Round 170: test_RMSE=1.1660, last round took 16.77s
Round 180: test_RMSE=1.16

# Full Pipeline: Which Cell Feeds What, and the Combined Mathematics

# Overview: what Cell 9 pulls from every other cell

Cell 9 is the orchestrator -- it does not introduce new logic itself, it calls functions built in every earlier cell, in sequence, every round.

| Cell | What it contributes to Cell 9 |
|---|---|
| Cell 1 | `device` -- routes every tensor/model to GPU or CPU |
| Cell 2 | `df` -- the raw loaded ratings (user_id, item_id, rating, timestamp) |
| Cell 3 | `train_df`, `test_df`, `n_users`, `n_items` -- re-indexed, split data |
| Cell 4 | `MatrixFactorization` class -- the model architecture itself |
| Cell 5 | NOT used in Cell 9 -- a separate, standalone centralized baseline (own model object, own training run, only used for RMSE comparison) |
| Cell 6 | `dirichlet_partition()` -- builds `client_data_full`, the non-IID split |
| Cell 7 | `local_train_dp_prox()` -- DP-SGD + FedProx local training, called once per client per round |
| Cell 8 | `generate_pairwise_masks()`, `client_mask_update()`, `secure_aggregate()` -- combines all clients' updates without exposing any individual one |
| (new, Cell 9) | `get_client_data_at_round()` -- temporal realism, using Cell 2's untouched timestamp column |

# The full workflow, step by step

1. Build the non-IID partition ONCE, before any rounds start (Cell 6's function, called from Cell 9):
$$\text{client\_data\_full} = \text{dirichlet\_partition}(\text{train\_df}, N_{\text{clients}}=10, \alpha=0.5)$$

2. Create ONE fresh, randomly-initialized global model (Cell 4's class, Cell 4's init logic):
$$\theta_{\text{global}}^{(0)} \sim \mathcal{N}(0, 0.01^2) \text{ (embeddings)}, \quad \theta_{\text{global}}^{(0)} = 0 \text{ (biases)}$$

3. For each of 300 rounds $t = 0, 1, \ldots, 299$:

   a. Snapshot the current global weights:
   $$\theta_{\text{global}}^{(t)} = \text{global\_model.state\_dict()}$$

   b. For each client $c = 0, \ldots, 9$:

      i. Determine how much of client $c$'s data has "arrived" by this round (temporal realism, using real timestamps):
      $$\text{cutoff}_t = t_{\min} + \frac{t+1}{300}(t_{\max} - t_{\min})$$
      $$D_c^{(t)} = \{(u,i,r,\tau) \in D_c : \tau \le \text{cutoff}_t\}$$

      ii. If $D_c^{(t)}$ is empty, skip this client this round.

      iii. Otherwise, run Cell 7's local training on $D_c^{(t)}$, starting from $\theta_{\text{global}}^{(t)}$ -- full math below.

   c. Combine all participating clients' updates via Cell 8's secure aggregation -- full math below.

   d. Update the global model:
   $$\theta_{\text{global}}^{(t+1)} = \text{secure\_aggregate}(\text{masked updates})$$

4. After all rounds, report final RMSE and the total privacy budget spent (epsilon).

# The combined mathematics, in full

# 1. Prediction (Cell 4)

$$\hat{r}_{u,i} = b_0 + b_u + b_i + \mathbf{p}_u \cdot \mathbf{q}_i$$

# 2. Local loss, combining task loss and FedProx (Cell 7)

For one example $(u,i,r)$, evaluated on client $c$'s currently-available data:

$$\mathcal{L}(\theta) = \underbrace{(\hat{r}_{u,i} - r)^2}_{\text{task loss}} + \underbrace{\frac{\mu}{2}\sum_{\text{all params}} (\theta - \theta_{\text{global}}^{(t)})^2}_{\text{FedProx proximal term}}$$

# 3. Per-sample gradient (Cell 7, via vmap + grad)

For each individual example $n$ in a batch:

$$g_n = \nabla_\theta \mathcal{L}_n(\theta)$$

Computed independently and in parallel for every example in the batch -- this is what vmap provides, since ordinary `.backward()` only yields one averaged gradient for the whole batch, not one per example.

# 4. Per-sample clipping (Cell 7, DP-SGD)

$$\|g_n\| = \sqrt{\sum_k g_{n,k}^2}, \qquad \bar{g}_n = g_n \cdot \min\left(1, \frac{C}{\|g_n\|}\right)$$

where $C$ is `clip_norm`. No single example's gradient can exceed norm $C$ after this step.

# 5. Noise addition (Cell 7, DP-SGD)

$$\tilde{g} = \sum_{n=1}^{B} \bar{g}_n + \mathcal{N}(0, \sigma^2), \qquad \sigma = \text{noise\_multiplier} \times C$$

where $B$ is the batch size. This is the actual differential privacy mechanism -- noise calibrated proportionally to the clipping bound $C$.

# 6. Manual parameter update (Cell 7 -- plain SGD, not Adam, since torch.func needs explicit params)

$$\theta \leftarrow \theta - \eta \cdot \frac{\tilde{g}}{B}$$

Repeated for every batch, across `local_epochs=5` passes over the client's currently-available data.

# 7. Client-side weighting and masking (Cell 8)

After local training produces $\theta_c$ (client $c$'s updated local weights) and $N_c$ (how many ratings it trained on):

$$w_c = \frac{N_c}{N}, \qquad N = \sum_{c} N_c$$

$$\text{masked}_c = w_c \cdot \theta_c + M_c$$

where $M_c$ is client $c$'s total pairwise mask contribution (built from `generate_pairwise_masks`):

$$M_c = \sum_{j>c} m_{cj} - \sum_{i<c} m_{ic}$$

# 8. Server-side aggregation (Cell 8)

$$\theta_{\text{global}}^{(t+1)} = \sum_{c} \text{masked}_c = \sum_c w_c \theta_c + \underbrace{\sum_c M_c}_{=\,0} = \sum_c \frac{N_c}{N}\theta_c$$

The masks cancel exactly (proven earlier), leaving exactly the FedAvg weighted average -- computed without the server ever seeing any individual $\theta_c$.

# 9. Privacy accounting, after all rounds (Cell 7's compute_epsilon, via RDP accountant)

$$\text{steps} = \text{local\_epochs} \times \text{rounds} \times \frac{N_c}{\text{batch\_size}}, \qquad q = \frac{\text{batch\_size}}{N_c}$$

The Renyi-DP accountant composes privacy loss across all these steps, converting to a final:

$$(\epsilon, \delta) \text{ guarantee, with } \delta = 10^{-5}$$

reported as the total privacy budget spent across the entire 300-round training run.

# Summary

Data flows from Cell 2/3 (loaded, cleaned, split) into Cell 6 (non-IID partitioned by Dirichlet) and Cell 4 (model architecture defined) -- Cell 9 then repeatedly calls Cell 7 (each client trains privately and briefly on its own, temporally-gated slice of data, protected by clipping + noise + FedProx's stability term) and Cell 8 (all clients' updates are combined via cancelling random masks, so the server only ever learns the correct total, never any individual contribution) -- producing, after 300 rounds, both a final accuracy number (RMSE) and a formal, quantified privacy guarantee (epsilon) describing exactly how much any single rating could have influenced the final model.